In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(patchwork)
  library(scales)
})
root <- getwd()
cm <- readRDS(file.path(root, "Affirmseq_cardiomyocyte.rds"))
fd <- cm@misc$figure_data
Idents(cm) <- "cc_merged"
cc <- c("0" = "#E41A1C", "1" = "#377EB8", "2" = "#4DAF4A", "3" = "#984EA3", "4" = "#FF7F00")
cond <- c("ctrl" = "#4EA3D8", "model" = "#E74C3C")
theme_set(theme_classic(base_size = 14))

In [ ]:
Idents(cm) <- "cell_class"
p_vln <- VlnPlot(cm, features = c("nFeature_RNA", "nCount_RNA"), ncol = 2, pt.size = 0, cols = "#F8766D") + theme(axis.text.x = element_text(angle = 45, hjust = 1, color = "black"), axis.text.y = element_text(color = "black"), axis.title = element_text(color = "black"), plot.title = element_text(face = "bold", hjust = 0.5))
ggsave(file.path(root, "Affirmseq_CM_Gene_UMI_violin.pdf"), p_vln, width = 4.8, height = 3.4, device = cairo_pdf)

u <- fd$umap
p_u1 <- ggplot(u, aes(umap_1, umap_2, color = cluster)) + geom_point(size = 0.45) + scale_color_manual(values = cc) + labs(x = "umap_1", y = "umap_2", color = NULL) + theme(legend.position = "right", axis.text = element_text(color = "black"))
p_u2 <- ggplot(u, aes(umap_1, umap_2, color = condition)) + geom_point(size = 0.45) + scale_color_manual(values = cond, labels = c(ctrl = "Normal", model = "GDM")) + labs(x = "umap_1", y = "umap_2", color = NULL) + theme(legend.position = "right", axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_cardiomyocyte_combined1.pdf"), p_u1 | p_u2, width = 7, height = 4, device = cairo_pdf)

cmp <- fd$composition_res08_merged %>% mutate(cluster = factor(cluster, levels = as.character(0:4)), condition = factor(condition, c("ctrl", "model")))
p_cmp <- ggplot(cmp, aes(cluster, prop, fill = condition)) + geom_col(width = 0.72, color = "white", linewidth = 0.25) + scale_fill_manual(values = cond) + scale_y_continuous(labels = percent, limits = c(0, 1)) + labs(x = "Cluster (res = 0.8, merged)", y = "Composition within cluster", fill = NULL) + theme(axis.text = element_text(color = "black"), legend.position = "top")
ggsave(file.path(root, "Affirmseq_Cluster_composition_res0.8.pdf"), p_cmp, width = 3.2, height = 4.2, device = cairo_pdf)

bp <- fd$bpplot %>% mutate(Condition = factor(condition, c("ctrl", "model")))
p_bp <- ggplot(bp, aes(Module_Score, reorder(Pathway, Module_Score), color = Condition)) + geom_line(aes(group = Pathway), color = "gray75", linewidth = 0.5) + geom_point(size = 3) + scale_color_manual(values = cond) + labs(title = "GO Pathway Enrichment Score Comparison", x = "Module Score", y = "Pathway") + theme(axis.text = element_text(color = "black"), axis.text.y = element_text(size = 8), panel.grid.major = element_line(color = "gray92", linewidth = 0.25))
ggsave(file.path(root, "Affirmseq_BPplot.pdf"), p_bp, width = 8.2, height = 4.2, device = cairo_pdf)

In [ ]:
or <- fd$cluster_or %>% mutate(cluster = factor(cluster, levels = as.character(0:4)))
p_or <- ggplot(or, aes(cluster, log2_or, color = cluster)) + geom_hline(yintercept = 0, linetype = "dashed") + geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.15, linewidth = 0.8) + geom_point(size = 3.5) + geom_text(aes(label = sig, y = upper + 0.15), color = "black", size = 4) + scale_color_manual(values = cc) + labs(x = "Clusters", y = "log2(Odds ratio) (GDM-exposed enrichment)") + theme(axis.text = element_text(color = "black"), legend.position = "none")
ggsave(file.path(root, "Affirmseq_Cluster_model_enrichment_log2OR.pdf"), p_or, width = 3.8, height = 4.6, device = cairo_pdf)

g <- fd$go
g3 <- g %>% filter(Group == "CC3_Downregulated", GO %in% c("energy derivation by oxidation of organic compounds", "generation of precursor metabolites and energy", "proton motive force-driven mitochondrial ATP synthesis", "proton motive force-driven ATP synthesis", "electron transport chain", "respiratory electron transport chain", "mitochondrial ATP synthesis coupled electron transport", "ATP synthesis coupled electron transport", "ATP biosynthetic process"))
g4 <- g %>% filter(Group == "CC4_Upregulated", GO %in% c("muscle contraction", "cardiac muscle contraction", "striated muscle contraction", "heart contraction", "circulatory system process", "striated muscle cell development", "regulation of heart rate", "heart process"))
bar_go <- function(x, title, fill) ggplot(x, aes(mlog10, reorder(GO, mlog10))) + geom_col(width = 0.7, fill = fill) + labs(title = title, x = expression(-log[10]("Adjusted P-value")), y = "GO terms") + theme(axis.text = element_text(color = "black"), axis.text.y = element_text(size = 8), plot.title = element_text(face = "bold", size = 12))
ggsave(file.path(root, "Affirmseq_GO_CC3down_CC4up_bar_FINAL.pdf"), bar_go(g3, "CC3 downregulated: mitochondrial energy metabolism", "#6FA8DC") | bar_go(g4, "CC4 upregulated: cardiac contractile programs", "#F05A4A"), width = 8.5, height = 3.8, device = cairo_pdf)

dot <- function(d) {
  d$features.plot <- factor(d$features.plot, levels = unique(d$features.plot))
  d$feature.groups <- factor(d$feature.groups, levels = unique(d$feature.groups))
  ggplot(d, aes(features.plot, factor(id), size = pct.exp, color = avg.exp.scaled)) + geom_point() + facet_grid(. ~ feature.groups, scales = "free_x", space = "free_x") + scale_color_gradient2(low = "#2166ac", mid = "white", high = "#b2182b", midpoint = 0, name = "Average Expression") + scale_size(range = c(0.4, 5.5), name = "Percent Expressed") + labs(x = "Features", y = "Identity") + theme(axis.text.x = element_text(angle = 60, hjust = 1, color = "black"), axis.text.y = element_text(color = "black"), strip.background = element_blank(), panel.spacing.x = unit(0.5, "lines"))
}
ggsave(file.path(root, "Affirmseq_pDotPlot_cardiomyocyte_combined.pdf"), dot(fd$dot_condition) / dot(fd$dot_cluster), width = 11, height = 8, device = cairo_pdf)